# 06 — Cluster Biological Interpretation

This notebook turns the marker-gene results from Notebook 04 into structured, dataset-grounded evidence for biological interpretation. It reuses the annotated AnnData object, saved marker statistics, cell-type annotations, and existing UMAP. It does **not** rerun normalization, clustering, UMAP, or differential-expression testing.

Outputs are written to `results/phase6/` for a later, separate literature-analysis phase.

In [1]:
from pathlib import Path
import warnings

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from IPython.display import Markdown, display

warnings.filterwarnings("ignore", category=FutureWarning)
sc.settings.verbosity = 2
sc.set_figure_params(dpi=100, facecolor="white", fontsize=10)
pd.set_option("display.max_colwidth", 100)

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "processed").is_dir() and (candidate / "results" / "tables").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the PBMC3k project root.")

PROJECT = find_project_root(Path.cwd().resolve())
PROCESSED = PROJECT / "data" / "processed"
TABLES = PROJECT / "results" / "tables"
FIGURES = PROJECT / "results" / "figures"
PHASE6 = PROJECT / "results" / "phase6"
PLOTS = PHASE6 / "plots"
PHASE6.mkdir(parents=True, exist_ok=True)
PLOTS.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT}")
print(f"Phase 6 output: {PHASE6}")

Project root: /Users/shelyjain/Desktop/Desktop - Shely's Macbook Pro/cosmos/26-the-backpropagators-analysis/PBMC3k
Phase 6 output: /Users/shelyjain/Desktop/Desktop - Shely's Macbook Pro/cosmos/26-the-backpropagators-analysis/PBMC3k/results/phase6


## 1. Inspect and reuse existing outputs

The inventory below makes the dependency chain explicit before any analysis. The most complete processed object is selected by preference (`phase3_annotated` before earlier phases), and all existing plots and tables are listed for traceability.

In [2]:
processed_objects = sorted(PROCESSED.glob("*.h5ad"))
existing_tables = sorted(TABLES.glob("*.csv"))
existing_plots = sorted(p for p in FIGURES.iterdir() if p.is_file())

preferred_objects = [
    PROCESSED / "pbmc3k_phase3_annotated.h5ad",
    PROCESSED / "pbmc3k_phase2_clustered.h5ad",
    PROCESSED / "pbmc3k_phase1_qc_top2000sd.h5ad",
]
adata_path = next((p for p in preferred_objects if p.exists()), None)
if adata_path is None:
    raise FileNotFoundError("No processed AnnData object was found.")

adata = ad.read_h5ad(adata_path)

inventory = pd.DataFrame(
    [
        ("Processed AnnData objects", ", ".join(p.name for p in processed_objects)),
        ("Object reused", adata_path.name),
        ("Shape", f"{adata.n_obs:,} cells × {adata.n_vars:,} selected genes"),
        ("Cluster labels", ", ".join(c for c in adata.obs.columns if c.startswith("leiden"))),
        ("Cell-type annotations", "cell_type" if "cell_type" in adata.obs else "missing"),
        ("Normalized expression", f"raw={adata.raw.shape if adata.raw is not None else None}; layers={list(adata.layers.keys())}"),
        ("Embeddings", ", ".join(adata.obsm.keys())),
        ("Saved marker outputs", ", ".join(p.name for p in existing_tables if "marker" in p.name or "annotation" in p.name)),
        ("Existing plots", ", ".join(p.name for p in existing_plots)),
    ],
    columns=["Reusable component", "Finding"],
)
display(inventory)

required_obs = {"leiden", "cell_type"}
missing_obs = required_obs.difference(adata.obs.columns)
if missing_obs:
    raise ValueError(f"Annotated object is missing required obs columns: {sorted(missing_obs)}")
if adata.raw is None:
    raise ValueError("The annotated object does not contain all-gene normalized expression in adata.raw.")
if "X_umap" not in adata.obsm:
    raise ValueError("The annotated object does not contain the existing UMAP embedding.")

cluster_counts = (
    adata.obs.groupby(["leiden", "cell_type"], observed=True)
    .size().rename("number_of_cells").reset_index()
)
display(cluster_counts)

,Reusable component,Finding
0,Processed AnnData objects,"pbmc3k_phase1_qc_top2000sd.h5ad, pbmc3k_phase2_clustered.h5ad, pbmc3k_phase3_annotated.h5ad"
1,Object reused,pbmc3k_phase3_annotated.h5ad
2,Shape,"2,638 cells × 2,000 selected genes"
3,Cluster labels,"leiden_0_3, leiden_0_4, leiden_0_5, leiden_0_6, leiden_0_7, leiden_0_8, leiden"
4,Cell-type annotations,cell_type
5,Normalized expression,"raw=(2638, 13656); layers=['counts', 'log_normalized']"
6,Embeddings,"X_pca, X_tsne, X_umap"
7,Saved marker outputs,"leiden_6_all_marker_statistics.csv, leiden_6_filtered_markers.csv, leiden_6_marker_summary.csv, ..."
8,Existing plots,".gitkeep, annotation_marker_dotplot.png, classification_class_balance.png, classification_confus..."


,leiden,cell_type,number_of_cells
0,0,Cytotoxic CD8 T cells,273
1,1,B cells,348
2,2,IL7R+ memory/helper T cells,602
3,3,Classical monocytes,502
4,4,CD16+ non-classical monocytes,171
5,5,NK cells,153
6,6,Activated/transitional T cells,128
7,7,Naive/resting T cells,450
8,8,Platelets,11


### Reuse decision

- `pbmc3k_phase3_annotated.h5ad` supplies cluster labels, cell types, all-gene normalized expression in `.raw`, counts/log-normalized layers, and the existing UMAP coordinates.
- `leiden_9_all_marker_statistics.csv` supplies Notebook 04's complete differential-expression output, including fold changes, adjusted p-values, and within/outside expression fractions.
- `leiden_9_cell_type_annotations.csv` supplies cluster-level labels, cell counts, evidence notes, and confidence.

Therefore no expensive upstream analysis is recomputed.

## 2. Load and standardize Notebook 04 marker genes

Equivalent Scanpy column names are mapped to the requested schema. Marker genes are regenerated from the existing AnnData ranking only if the saved CSV is missing; this fallback does not rerun differential-expression testing.

In [3]:
marker_path = TABLES / "leiden_9_all_marker_statistics.csv"
annotation_path = TABLES / "leiden_9_cell_type_annotations.csv"

if marker_path.exists():
    marker_source = pd.read_csv(marker_path)
    marker_origin = marker_path.name
elif "rank_genes_groups_leiden" in adata.uns:
    marker_source = sc.get.rank_genes_groups_df(adata, group=None, key="rank_genes_groups_leiden")
    marker_origin = "adata.uns['rank_genes_groups_leiden'] fallback"
else:
    raise FileNotFoundError("Notebook 04 marker CSV and stored marker ranking are both missing.")

annotations = pd.read_csv(annotation_path) if annotation_path.exists() else cluster_counts.rename(
    columns={"leiden": "cluster", "number_of_cells": "cell_count"}
)

column_aliases = {
    "group": "cluster",
    "names": "gene",
    "logfoldchanges": "avg_log2FC",
    "pvals_adj": "adjusted_p_value",
    "pct_nz_group": "pct_in",
    "fraction_expressing_in_cluster": "pct_in",
    "pct_nz_reference": "pct_out",
    "fraction_expressing_outside_cluster": "pct_out",
}

# Rename only the first available alias for each standardized field.
rename_map = {}
claimed_targets = set(marker_source.columns)
for source, target in column_aliases.items():
    if source in marker_source.columns and target not in claimed_targets:
        rename_map[source] = target
        claimed_targets.add(target)
markers = marker_source.rename(columns=rename_map).copy()

required_marker_columns = ["cluster", "gene", "avg_log2FC", "adjusted_p_value", "pct_in", "pct_out"]
missing_marker_columns = [c for c in required_marker_columns if c not in markers.columns]
if missing_marker_columns:
    raise ValueError(f"Marker table cannot be standardized; missing {missing_marker_columns}")

markers["cluster"] = markers["cluster"].astype(str)
annotations = annotations.rename(columns={"leiden": "cluster"}).copy()
annotations["cluster"] = annotations["cluster"].astype(str)
annotation_columns = [c for c in ["cluster", "cell_type", "cell_count", "evidence_notes", "confidence"] if c in annotations]
markers = markers.drop(columns=["cell_type", "cell_count", "confidence"], errors="ignore").merge(
    annotations[annotation_columns], on="cluster", how="left", validate="many_to_one"
)

for col in ["avg_log2FC", "adjusted_p_value", "pct_in", "pct_out"]:
    markers[col] = pd.to_numeric(markers[col], errors="coerce")

standardized_columns = [
    "cluster", "cell_type", "gene", "avg_log2FC", "adjusted_p_value", "pct_in", "pct_out"
]
print(f"Loaded {len(markers):,} marker rows from {marker_origin}.")
display(markers[standardized_columns].head())

Loaded 122,904 marker rows from leiden_9_all_marker_statistics.csv.


,cluster,cell_type,gene,avg_log2FC,adjusted_p_value,pct_in,pct_out
0,0,Cytotoxic CD8 T cells,CCL5,5.365429,2.447579e-111,0.945055,0.243975
1,0,Cytotoxic CD8 T cells,NKG7,5.149965,2.524331e-104,0.959707,0.222410
2,0,Cytotoxic CD8 T cells,B2M,0.759635,1.219478e-70,1.000000,0.999577
3,0,Cytotoxic CD8 T cells,GZMA,4.028471,5.997373e-68,0.787546,0.129387
4,0,Cytotoxic CD8 T cells,CST7,4.054892,1.801775e-67,0.776557,0.129387


## 3. Rank marker genes with a composite score

$$\text{Marker Score} = \text{avg log2FC} \times (\text{pct in} - \text{pct out}) \times \min[-\log_{10}(\text{adjusted p-value}), 20]$$

The exported table retains the fold change, both expression fractions, the specificity difference, the uncapped and capped significance terms, and the final score. Extremely small or zero p-values are safely bounded at the smallest positive floating-point value before taking the logarithm.

In [4]:
tiny = np.finfo(float).tiny
markers["specificity_delta"] = markers["pct_in"] - markers["pct_out"]
markers["neg_log10_adjusted_p_value"] = -np.log10(markers["adjusted_p_value"].clip(lower=tiny))
markers["capped_neg_log10_adjusted_p_value"] = markers["neg_log10_adjusted_p_value"].clip(upper=20)
markers["marker_score"] = (
    markers["avg_log2FC"]
    * markers["specificity_delta"]
    * markers["capped_neg_log10_adjusted_p_value"]
)
markers["rank_within_cluster"] = markers.groupby("cluster")["marker_score"].rank(
    method="first", ascending=False, na_option="bottom"
).astype("Int64")

front_columns = standardized_columns + [
    "specificity_delta",
    "neg_log10_adjusted_p_value",
    "capped_neg_log10_adjusted_p_value",
    "marker_score",
    "rank_within_cluster",
]
ranked_markers = markers[front_columns + [c for c in markers.columns if c not in front_columns]].sort_values(
    ["cluster", "marker_score"], ascending=[True, False], kind="stable"
)
ranked_path = PHASE6 / "ranked_marker_genes.csv"
ranked_markers.to_csv(ranked_path, index=False)
print(f"Saved {len(ranked_markers):,} ranked marker rows to {ranked_path.relative_to(PROJECT)}")
display(ranked_markers[front_columns].groupby("cluster", sort=True).head(3))

Saved 122,904 ranked marker rows to results/phase6/ranked_marker_genes.csv


,cluster,cell_type,gene,avg_log2FC,adjusted_p_value,pct_in,pct_out,specificity_delta,neg_log10_adjusted_p_value,capped_neg_log10_adjusted_p_value,marker_score,rank_within_cluster
1,0,Cytotoxic CD8 T cells,NKG7,5.149965,2.524331e-104,0.959707,0.222410,0.737297,103.597854,20.000000,75.941055,1
0,0,Cytotoxic CD8 T cells,CCL5,5.365429,2.447579e-111,0.945055,0.243975,0.701080,110.611263,20.000000,75.231933,2
3,0,Cytotoxic CD8 T cells,GZMA,4.028471,5.997373e-68,0.787546,0.129387,0.658159,67.222039,20.000000,53.027480,3
13657,1,B cells,CD79A,7.704485,1.223091e-164,0.925287,0.041485,0.883803,163.912541,20.000000,136.184875,1
13662,1,B cells,MS4A1,6.358171,2.358891e-133,0.841954,0.053275,0.788679,132.627292,20.000000,100.291108,2
13659,1,B cells,CD79B,5.473424,1.834737e-149,0.905172,0.141921,0.763251,148.736426,20.000000,83.551929,3
40958,2,IL7R+ memory/helper T cells,HLA-DRB5,-3.740428,4.883566e-43,0.073090,0.433694,-0.360604,42.311263,20.000000,26.976249,1
40957,2,IL7R+ memory/helper T cells,FCER1G,-3.889404,2.750160e-42,0.094684,0.436149,-0.341465,41.560642,20.000000,26.561903,2
40960,2,IL7R+ memory/helper T cells,TYROBP,-3.859096,4.386386e-45,0.144518,0.472495,-0.327977,44.357893,20.000000,25.313879,3
40970,3,Classical monocytes,S100A8,7.360887,7.621623e-225,0.942231,0.120318,0.821913,224.117953,20.000000,121.000134,1


## 4. Select representative genes

A representative marker must be statistically supported, more common inside than outside its cluster, and expressed by enough cells to reduce sensitivity to rare-cell outliers. The primary eligibility rules are:

- positive average log2 fold change;
- positive within-minus-outside expression fraction;
- adjusted p-value ≤ 0.05;
- expression in at least 20% of cells in the cluster.

Within that evidence set, the composite marker score selects ten genes per cluster. If a cluster has fewer than ten primary candidates, the same evidence rules are relaxed only for the 20% prevalence threshold and the output records which tier was used.

In [5]:
N_REPRESENTATIVE = 10
base_evidence = (
    (ranked_markers["avg_log2FC"] > 0)
    & (ranked_markers["specificity_delta"] > 0)
    & (ranked_markers["adjusted_p_value"] <= 0.05)
)
primary = ranked_markers[base_evidence & (ranked_markers["pct_in"] >= 0.20)].copy()
primary["selection_tier"] = "primary: pct_in >= 0.20"

selected_parts = []
for cluster, cluster_rows in ranked_markers.groupby("cluster", sort=True):
    chosen = primary[primary["cluster"] == cluster].head(N_REPRESENTATIVE).copy()
    if len(chosen) < N_REPRESENTATIVE:
        selected_genes = set(chosen["gene"])
        fallback = ranked_markers[
            (ranked_markers["cluster"] == cluster)
            & base_evidence
            & ~ranked_markers["gene"].isin(selected_genes)
        ].head(N_REPRESENTATIVE - len(chosen)).copy()
        fallback["selection_tier"] = "fallback: positive, specific, adjusted p <= 0.05"
        chosen = pd.concat([chosen, fallback], ignore_index=True)
    chosen["representative_rank"] = np.arange(1, len(chosen) + 1)
    selected_parts.append(chosen)

selected = pd.concat(selected_parts, ignore_index=True)
selected_columns = [
    "cluster", "cell_type", "representative_rank", "gene", "avg_log2FC",
    "adjusted_p_value", "pct_in", "pct_out", "specificity_delta",
    "capped_neg_log10_adjusted_p_value", "marker_score", "selection_tier",
]
selected = selected[selected_columns]
selected_path = PHASE6 / "selected_marker_genes.csv"
selected.to_csv(selected_path, index=False)

print(f"Saved {len(selected):,} representative markers to {selected_path.relative_to(PROJECT)}")
display(
    selected.groupby(["cluster", "cell_type"], sort=True)
    .agg(number_selected=("gene", "size"), genes=("gene", lambda x: ", ".join(x)))
)

Saved 90 representative markers to results/phase6/selected_marker_genes.csv


,,number_selected,genes
cluster,cell_type,,
0,Cytotoxic CD8 T cells,10,"NKG7, CCL5, GZMA, CST7, GZMK, CTSW, CD8A, LYAR, GZMH, KLRG1"
1,B cells,10,"CD79A, MS4A1, CD79B, TCL1A, HLA-DQA1, LINC00926, VPREB3, HLA-DQB1, HLA-DRA, FCER2"
2,IL7R+ memory/helper T cells,10,"IL32, IL7R, CD3D, LTB, CD3E, CD2, AQP3, LDHB, TRAT1, SPOCK2"
3,Classical monocytes,10,"S100A8, LGALS2, S100A9, FCN1, CST3, TYROBP, CD14, MS4A6A, LST1, AIF1"
4,CD16+ non-classical monocytes,10,"FCGR3A, IFITM3, MS4A7, RP11-290F20.3, LST1, FCER1G, AIF1, SERPINA1, CDKN1C, CFD"
5,NK cells,10,"GZMB, FGFBP2, GNLY, PRF1, NKG7, CST7, SPON2, GZMA, CCL4, CTSW"
6,Activated/transitional T cells,10,"CCL5, IL32, CD3D, RPL23A, RPS3, RPS12, MALAT1, RPS25, RPS14, RPL13"
7,Naive/resting T cells,10,"CCR7, CD3D, LDHB, PRKCQ-AS1, NOSIP, CD7, CD3E, PIK3IP1, LEF1, C6orf48"
8,Platelets,10,"PPBP, PF4, GNG11, SDPR, SPARC, CD9, GP9, ITGA2B, HIST1H2AC, NRGN"


## 5. Recommend one pilot cluster

The automatic recommendation balances marker clarity, specificity, expression consistency, annotation confidence, and sample size. Candidate clusters must have at least 100 cells and a high-confidence annotation. A clarity score summarizes the selected markers, and a capped cell sufficiency factor prevents very small clusters from winning solely because of high specificity.

In [6]:
pilot_candidates = (
    selected.groupby(["cluster", "cell_type"], as_index=False)
    .agg(
        mean_marker_score=("marker_score", "mean"),
        median_specificity=("specificity_delta", "median"),
        median_pct_in=("pct_in", "median"),
    )
    .merge(annotations[[c for c in ["cluster", "cell_count", "confidence"] if c in annotations]], on="cluster", how="left")
)
pilot_candidates["clarity_score"] = (
    np.log1p(pilot_candidates["mean_marker_score"].clip(lower=0))
    + 2 * pilot_candidates["median_specificity"]
    + pilot_candidates["median_pct_in"]
)
pilot_candidates["cell_sufficiency"] = np.minimum(pilot_candidates["cell_count"] / 150, 1.0)
pilot_candidates["confidence_eligible"] = pilot_candidates["confidence"].str.lower().eq("high")
pilot_candidates["sample_size_eligible"] = pilot_candidates["cell_count"] >= 100
pilot_candidates["pilot_score"] = (
    pilot_candidates["clarity_score"]
    * pilot_candidates["cell_sufficiency"]
    * pilot_candidates["confidence_eligible"].astype(float)
)
eligible_pilots = pilot_candidates[
    pilot_candidates["confidence_eligible"] & pilot_candidates["sample_size_eligible"]
]
if eligible_pilots.empty:
    raise ValueError("No cluster met the pilot eligibility criteria.")

pilot = eligible_pilots.sort_values("pilot_score", ascending=False).iloc[0]
pilot_cluster = str(pilot["cluster"])
pilot_cell_type = str(pilot["cell_type"])
pilot_cell_count = int(pilot["cell_count"])
pilot_markers = selected[selected["cluster"] == pilot_cluster].sort_values("representative_rank")
pilot_genes = pilot_markers["gene"].tolist()

display(pilot_candidates.sort_values("pilot_score", ascending=False).round(3))
display(Markdown(
    f"**Recommended pilot: cluster {pilot_cluster} — {pilot_cell_type}.** "
    f"It has {pilot_cell_count:,} cells, a {str(pilot['confidence']).lower()}-confidence annotation, "
    f"a median selected-marker specificity of {pilot['median_specificity']:.3f}, and median within-cluster "
    f"expression of {pilot['median_pct_in']:.3f}. It ranked highest among eligible clusters after balancing "
    "marker clarity with cell-count sufficiency."
))

,cluster,cell_type,mean_marker_score,median_specificity,median_pct_in,cell_count,confidence,clarity_score,cell_sufficiency,confidence_eligible,sample_size_eligible,pilot_score
5,5,NK cells,99.782,0.792,0.961,153,high,7.157,1.000,True,True,7.157
3,3,Classical monocytes,88.811,0.740,0.950,502,high,6.927,1.000,True,True,6.927
4,4,CD16+ non-classical monocytes,71.378,0.741,0.956,171,high,6.721,1.000,True,True,6.721
1,1,B cells,80.337,0.653,0.849,348,high,6.553,1.000,True,True,6.553
0,0,Cytotoxic CD8 T cells,45.934,0.554,0.683,273,high,5.641,1.000,True,True,5.641
2,2,IL7R+ memory/helper T cells,17.783,0.382,0.794,602,high,4.491,1.000,True,True,4.491
8,8,Platelets,57.052,0.975,1.000,11,high,7.011,0.073,True,False,0.514
6,6,Activated/transitional T cells,0.557,0.006,1.000,128,moderate,1.454,0.853,False,True,0.000
7,7,Naive/resting T cells,11.862,0.323,0.667,450,moderate,3.868,1.000,False,True,0.000


**Recommended pilot: cluster 5 — NK cells.** It has 153 cells, a high-confidence annotation, a median selected-marker specificity of 0.792, and median within-cluster expression of 0.961. It ranked highest among eligible clusters after balancing marker clarity with cell-count sufficiency.

## 6. Validation figures for the pilot cluster

All plots use `adata.raw` (the existing all-gene normalized expression). The feature plot uses `adata.obsm['X_umap']`; no embedding is recalculated.

In [7]:
available_genes = [g for g in pilot_genes if g in adata.raw.var_names]
if len(available_genes) < 4:
    raise ValueError("Too few selected pilot genes are available in adata.raw for validation plots.")

# Dot plot: prevalence (dot size) and average expression (color) across all clusters.
dot = sc.pl.dotplot(
    adata, available_genes, groupby="leiden", use_raw=True,
    standard_scale="var", return_fig=True, show=False,
    title=f"Pilot markers across clusters — {pilot_cell_type}",
)
dot_path = PLOTS / "pilot_cluster_dotplot.png"
dot.savefig(dot_path, dpi=180, bbox_inches="tight")
plt.close("all")

# Heatmap: normalized expression patterns using the existing cluster ordering.
sc.pl.heatmap(
    adata, available_genes, groupby="leiden", use_raw=True,
    standard_scale="var", swap_axes=True, show=False,
    figsize=(10, 6), cmap="viridis",
)
heatmap_path = PLOTS / "pilot_cluster_heatmap.png"
plt.gcf().suptitle(f"Pilot marker heatmap — {pilot_cell_type}", y=1.02)
plt.savefig(heatmap_path, dpi=180, bbox_inches="tight")
plt.close("all")

# Violin plot: direct pilot-versus-all-other-cells expression comparison.
adata.obs["pilot_status"] = np.where(
    adata.obs["leiden"].astype(str).eq(pilot_cluster),
    f"Cluster {pilot_cluster}: {pilot_cell_type}",
    "All other clusters",
)
adata.obs["pilot_status"] = pd.Categorical(
    adata.obs["pilot_status"],
    categories=["All other clusters", f"Cluster {pilot_cluster}: {pilot_cell_type}"],
)
sc.pl.violin(
    adata, available_genes[:4], groupby="pilot_status", use_raw=True,
    rotation=20, stripplot=False, multi_panel=True, show=False,
)
violin_path = PLOTS / "pilot_cluster_violin.png"
plt.gcf().suptitle(f"Pilot versus other cells — {pilot_cell_type}", y=1.04)
plt.savefig(violin_path, dpi=180, bbox_inches="tight")
plt.close("all")

# Feature plot: selected genes on the UMAP computed in Notebook 03.
sc.pl.umap(
    adata, color=available_genes[:4], use_raw=True, ncols=2,
    frameon=False, color_map="magma", show=False,
)
feature_path = PLOTS / "pilot_cluster_umap_featureplot.png"
plt.gcf().suptitle(f"Pilot markers on existing UMAP — {pilot_cell_type}", y=1.02)
plt.savefig(feature_path, dpi=180, bbox_inches="tight")
plt.close("all")

plot_outputs = [dot_path, heatmap_path, violin_path, feature_path]
print("Saved validation figures:")
for path in plot_outputs:
    print(f"- {path.relative_to(PROJECT)}")

/Users/shelyjain/Desktop/Desktop - Shely's Macbook Pro/cosmos/.venv/lib/python3.12/site-packages/scipy/stats/_kde.py:229: RuntimeWarning: divide by zero encountered in vecdot
  self._neff = 1/np.vecdot(self._weights, self._weights)
/Users/shelyjain/Desktop/Desktop - Shely's Macbook Pro/cosmos/.venv/lib/python3.12/site-packages/scipy/stats/_kde.py:229: RuntimeWarning: overflow encountered in vecdot
  self._neff = 1/np.vecdot(self._weights, self._weights)
/Users/shelyjain/Desktop/Desktop - Shely's Macbook Pro/cosmos/.venv/lib/python3.12/site-packages/scipy/stats/_kde.py:229: RuntimeWarning: invalid value encountered in vecdot
  self._neff = 1/np.vecdot(self._weights, self._weights)
/Users/shelyjain/Desktop/Desktop - Shely's Macbook Pro/cosmos/.venv/lib/python3.12/site-packages/scipy/stats/_kde.py:229: RuntimeWarning: divide by zero encountered in vecdot
  self._neff = 1/np.vecdot(self._weights, self._weights)
/Users/shelyjain/Desktop/Desktop - Shely's Macbook Pro/cosmos/.venv/lib/python3

Saved validation figures:
- results/phase6/plots/pilot_cluster_dotplot.png
- results/phase6/plots/pilot_cluster_heatmap.png
- results/phase6/plots/pilot_cluster_violin.png
- results/phase6/plots/pilot_cluster_umap_featureplot.png


## 7. Create the pilot-cluster summary

The report below is generated only from this dataset's cell labels, marker statistics, and expression patterns. The short biological interpretation is a cautious inference from the coordinated marker program; it is not a literature review or a disease claim.

In [8]:
def markdown_table(frame: pd.DataFrame) -> str:
    """Render a small DataFrame as Markdown without requiring tabulate."""
    shown = frame.copy()
    headers = [str(c) for c in shown.columns]
    rows = [[str(v) for v in row] for row in shown.itertuples(index=False, name=None)]
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    lines.extend("| " + " | ".join(row) + " |" for row in rows)
    return "\n".join(lines)

report_table = pilot_markers[
    ["representative_rank", "gene", "avg_log2FC", "adjusted_p_value", "pct_in", "pct_out", "specificity_delta", "marker_score"]
].copy()
report_table.columns = ["Rank", "Gene", "avg_log2FC", "Adjusted p", "pct_in", "pct_out", "Specificity", "Marker score"]
for col in ["avg_log2FC", "pct_in", "pct_out", "Specificity", "Marker score"]:
    report_table[col] = report_table[col].map(lambda x: f"{x:.3f}")
report_table["Adjusted p"] = report_table["Adjusted p"].map(lambda x: f"{x:.2e}")

top_gene = pilot_markers.iloc[0]
median_specificity = pilot_markers["specificity_delta"].median()
median_pct_in = pilot_markers["pct_in"].median()
top_genes_text = ", ".join(f"`{g}`" for g in pilot_genes)

role_templates = {
    "NK cells": (
        "The coordinated high and cluster-specific expression of cytotoxic-effector genes such as GNLY, GZMB, "
        "PRF1, NKG7, and FGFBP2 suggests that these cells are equipped for rapid target-cell killing. In this dataset, "
        "that expression program supports the assigned NK-cell identity and distinguishes the cluster from the T-cell "
        "clusters, even where shared genes such as NKG7 or CCL5 occur."
    ),
    "B cells": (
        "The coordinated expression of B-lineage receptor and antigen-presentation genes suggests a population centered "
        "on B-cell recognition and immune communication. This interpretation is limited to the expression program observed here."
    ),
    "Classical monocytes": (
        "The coordinated expression of myeloid and inflammatory-response markers suggests a circulating innate immune "
        "population with strong sensing and effector capacity in this dataset."
    ),
    "CD16+ non-classical monocytes": (
        "The coordinated FCGR3A-associated myeloid expression program suggests a distinct monocyte state, separated from "
        "the classical-monocyte cluster by its marker balance in this dataset."
    ),
}
biological_interpretation = role_templates.get(
    pilot_cell_type,
    "The coordinated expression and specificity of the selected markers support the assigned immune-cell identity and suggest a distinct functional state within this dataset."
)

future_questions = []
for gene in pilot_genes[:4]:
    future_questions.extend([
        f"- What is the biological function of {gene}?",
        f"- Is {gene} a known marker of {pilot_cell_type}?",
        f"- Which immune pathways involve {gene}?",
        f"- Has {gene} been associated with diseases, and in what experimental contexts?",
    ])

report = f"""# Cluster {pilot_cluster}: {pilot_cell_type}

## Assigned Cell Type

{pilot_cell_type} (Notebook 04 annotation confidence: {str(pilot['confidence']).lower()}).

## Number of Cells

{pilot_cell_count:,} of {adata.n_obs:,} cells ({100 * pilot_cell_count / adata.n_obs:.1f}%).

## Top Marker Genes

{top_genes_text}

## Marker Ranking Table

{markdown_table(report_table)}

## What Makes This Cluster Different?

The selected markers have a median within-cluster expression fraction of {median_pct_in:.3f} and a median specificity difference of {median_specificity:.3f}. The top-ranked gene, {top_gene['gene']}, is detected in {top_gene['pct_in']:.1%} of this cluster versus {top_gene['pct_out']:.1%} outside it, with an average log2 fold change of {top_gene['avg_log2FC']:.2f}. Across the full selected panel, positive fold changes, adjusted significance, broad within-cluster detection, and higher expression prevalence inside the cluster jointly distinguish this group; the conclusion does not rely on fold change alone.

## Biological Interpretation

{biological_interpretation}

This is a dataset-grounded interpretation, not a literature-supported conclusion. No external sources were used in this notebook.

## Questions for Future Literature Search

{chr(10).join(future_questions)}

## Limitations

This single PBMC3k expression dataset and its unsupervised clusters cannot determine:

- disease diagnosis;
- donor identity;
- race;
- ethnicity;
- personality.

Cluster labels and biological interpretations are analytical assignments, not direct measurements of those traits. The smallest populations are especially sensitive to sampling variability, and marker expression can overlap across related immune-cell states.
"""

summary_path = PHASE6 / "pilot_cluster_summary.md"
summary_path.write_text(report, encoding="utf-8")
print(f"Saved report to {summary_path.relative_to(PROJECT)}")
display(Markdown(report))

Saved report to results/phase6/pilot_cluster_summary.md


# Cluster 5: NK cells

## Assigned Cell Type

NK cells (Notebook 04 annotation confidence: high).

## Number of Cells

153 of 2,638 cells (5.8%).

## Top Marker Genes

`GZMB`, `FGFBP2`, `GNLY`, `PRF1`, `NKG7`, `CST7`, `SPON2`, `GZMA`, `CCL4`, `CTSW`

## Marker Ranking Table

| Rank | Gene | avg_log2FC | Adjusted p | pct_in | pct_out | Specificity | Marker score |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | GZMB | 7.891 | 4.62e-85 | 0.974 | 0.068 | 0.906 | 142.959 |
| 2 | FGFBP2 | 6.923 | 1.18e-70 | 0.895 | 0.062 | 0.834 | 115.462 |
| 3 | GNLY | 7.359 | 2.05e-72 | 0.915 | 0.135 | 0.780 | 114.827 |
| 4 | PRF1 | 6.527 | 9.81e-83 | 0.967 | 0.106 | 0.861 | 112.412 |
| 5 | NKG7 | 6.985 | 4.39e-88 | 1.000 | 0.256 | 0.744 | 104.006 |
| 6 | CST7 | 5.520 | 6.50e-76 | 0.967 | 0.149 | 0.818 | 90.354 |
| 7 | SPON2 | 6.330 | 5.11e-48 | 0.739 | 0.039 | 0.699 | 88.511 |
| 8 | GZMA | 5.479 | 2.62e-74 | 0.954 | 0.151 | 0.803 | 88.033 |
| 9 | CCL4 | 5.431 | 4.90e-45 | 0.745 | 0.075 | 0.670 | 72.808 |
| 10 | CTSW | 4.689 | 1.01e-75 | 0.987 | 0.257 | 0.730 | 68.444 |

## What Makes This Cluster Different?

The selected markers have a median within-cluster expression fraction of 0.961 and a median specificity difference of 0.792. The top-ranked gene, GZMB, is detected in 97.4% of this cluster versus 6.8% outside it, with an average log2 fold change of 7.89. Across the full selected panel, positive fold changes, adjusted significance, broad within-cluster detection, and higher expression prevalence inside the cluster jointly distinguish this group; the conclusion does not rely on fold change alone.

## Biological Interpretation

The coordinated high and cluster-specific expression of cytotoxic-effector genes such as GNLY, GZMB, PRF1, NKG7, and FGFBP2 suggests that these cells are equipped for rapid target-cell killing. In this dataset, that expression program supports the assigned NK-cell identity and distinguishes the cluster from the T-cell clusters, even where shared genes such as NKG7 or CCL5 occur.

This is a dataset-grounded interpretation, not a literature-supported conclusion. No external sources were used in this notebook.

## Questions for Future Literature Search

- What is the biological function of GZMB?
- Is GZMB a known marker of NK cells?
- Which immune pathways involve GZMB?
- Has GZMB been associated with diseases, and in what experimental contexts?
- What is the biological function of FGFBP2?
- Is FGFBP2 a known marker of NK cells?
- Which immune pathways involve FGFBP2?
- Has FGFBP2 been associated with diseases, and in what experimental contexts?
- What is the biological function of GNLY?
- Is GNLY a known marker of NK cells?
- Which immune pathways involve GNLY?
- Has GNLY been associated with diseases, and in what experimental contexts?
- What is the biological function of PRF1?
- Is PRF1 a known marker of NK cells?
- Which immune pathways involve PRF1?
- Has PRF1 been associated with diseases, and in what experimental contexts?

## Limitations

This single PBMC3k expression dataset and its unsupervised clusters cannot determine:

- disease diagnosis;
- donor identity;
- race;
- ethnicity;
- personality.

Cluster labels and biological interpretations are analytical assignments, not direct measurements of those traits. The smallest populations are especially sensitive to sampling variability, and marker expression can overlap across related immune-cell states.


## Output manifest and reproducibility checks

In [9]:
expected_outputs = [ranked_path, selected_path, summary_path, *plot_outputs]
manifest = pd.DataFrame({
    "output": [str(p.relative_to(PROJECT)) for p in expected_outputs],
    "exists": [p.exists() for p in expected_outputs],
    "size_bytes": [p.stat().st_size if p.exists() else 0 for p in expected_outputs],
})
display(manifest)

assert manifest["exists"].all(), "At least one required Phase 6 output is missing."
assert selected.groupby("cluster").size().between(8, 12).all(), "Each cluster should have approximately 8–12 markers."
assert len(selected["cluster"].unique()) == adata.obs["leiden"].nunique(), "Not all clusters are represented."
assert np.allclose(
    ranked_markers["marker_score"],
    ranked_markers["avg_log2FC"] * ranked_markers["specificity_delta"] * ranked_markers["capped_neg_log10_adjusted_p_value"],
    equal_nan=True,
), "Marker-score components do not reproduce the exported score."
print("All Phase 6 checks passed.")

,output,exists,size_bytes
0,results/phase6/ranked_marker_genes.csv,True,37956948
1,results/phase6/selected_marker_genes.csv,True,15114
2,results/phase6/pilot_cluster_summary.md,True,3434
3,results/phase6/plots/pilot_cluster_dotplot.png,True,66733
4,results/phase6/plots/pilot_cluster_heatmap.png,True,68302
5,results/phase6/plots/pilot_cluster_violin.png,True,123249
6,results/phase6/plots/pilot_cluster_umap_featureplot.png,True,336593


All Phase 6 checks passed.
